Process sessions
Jira: https://keyless.atlassian.net/browse/BIOM-625

In [ ]:
import numpy as np
from loguru import logger

import datasets
import os

import modules.globals
from modules import core

from tqdm.auto import tqdm

import pandas as pd

In [ ]:
ds_root = "s3://sagemaker-production-eu-central-1-kl-biometric-datasets/raw_datasets/face_biometrics/deepfakes/hackathon_2025-07_deepfakes/to_process_to_create_dest_dataset/"
ds_key = "train_aggregation_ensemble/part_0"
output_ds_root = "s3://sagemaker-production-eu-central-1-kl-biometric-datasets/raw_datasets/face_biometrics/deepfakes/hackathon_2025-07_deepfakes/destination_datasets/"

local_original_imgs_root = "/home/sagemaker-user/face_swap/original_images/"
local_swapped_imgs_root = "/home/sagemaker-user/face_swap/swapped_images_enh/"
execution_provider = "cuda"  # cuda or cpu
face_enhancer = True
rng_seed = 42

In [ ]:
### init ###
modules.globals.execution_providers = core.decode_execution_providers(
    [execution_provider]
)
frame_processors = ["face_swapper"]
modules.globals.fp_ui["face_enhancer"] = False
if face_enhancer:
    frame_processors.append("face_enhancer")
    modules.globals.fp_ui["face_enhancer"] = True
np.random.seed(rng_seed)

modules.globals.max_memory = core.suggest_max_memory()
modules.globals.execution_threads = core.suggest_execution_threads()
core.limit_resources()

In [ ]:
ds_path = os.path.join(ds_root, ds_key, "hf_dataset")
ds = datasets.Dataset.load_from_disk(ds_path)


In [ ]:
light_ds = ds.remove_columns(["img_raw", "source_img_raw"])
metadata_df = light_ds.to_pandas()

In [ ]:
local_swapped_imgs_path = os.path.join(local_swapped_imgs_root, ds_key)
os.makedirs(local_swapped_imgs_path, exist_ok=True)

In [ ]:
progress_df_path = os.path.join(local_swapped_imgs_path, "progress.parquet")
if os.path.exists(progress_df_path):
    progress_df = pd.read_parquet(progress_df_path)
    sessions_completed = progress_df[progress_df["processed"] == True]
    print(
        f"Completed {len(sessions_completed)} out of {len(progress_df)} sessions already"
    )
else:
    progress_df = pd.DataFrame(
        {
            "session_folder": pd.unique(metadata_df["session_folder"]),
            "processed": [False] * len(pd.unique(metadata_df["session_folder"])),
        }
    )
    progress_df = progress_df.set_index("session_folder")

In [ ]:
session_img_paths = {}
for row in tqdm(ds):
    session_folder = row["session_folder"]
    session_local_path = os.path.join(local_swapped_imgs_path, session_folder)

    if session_folder not in session_img_paths:
        session_img_paths[session_folder] = []
        os.makedirs(session_local_path, exist_ok=True)
        source_img_path = os.path.join(session_local_path, "source_img.jpg")
        row["source_img_raw"].save(source_img_path)

    img_path = os.path.join(session_local_path, row["photo_name"])
    if not img_path.endswith == ".jpg":
        img_path += ".jpg"
    row["img_raw"].save(img_path)

    session_img_paths[session_folder].append(img_path)

In [ ]:
session_ds_list = []
for session_folder in tqdm(session_img_paths.keys()):
    if not progress_df.loc[session_folder]["processed"]:
        logger.info(f"Processing session {session_folder}")
        try:
            source_img_path = os.path.join(
                os.path.dirname(session_img_paths[session_folder][0]), "source_img.jpg"
            )
            for frame_processor in core.get_frame_processors_modules(frame_processors):
                logger.info(f"Progressing... {frame_processor.NAME}")
                print(f"Total frames: {len(session_img_paths[session_folder])}")
                frame_processor.process_video(
                    source_img_path, session_img_paths[session_folder]
                )
                core.release_resources()
            progress_df.loc[session_folder, "processed"] = True
            progress_df.to_parquet(progress_df_path)
        except Exception as e:
            print(f"error in session {session_folder}: {e}")
            continue

    session_df = metadata_df[metadata_df.session_folder == session_folder].copy()
    # session_df.loc[:, "img_raw"] = session_img_paths[session_folder]
    session_path = os.path.join(local_swapped_imgs_path, session_folder)
    session_df.loc[:, "img_raw"] = session_df["photo_name"].apply(lambda x: os.path.join(session_path, x+".jpg"))
    session_df.loc[:, "attack_type"] = "deepfake"
    session_ds = datasets.Dataset.from_pandas(session_df, preserve_index=False)
    session_ds = session_ds.cast_column("img_raw", datasets.Image())
    session_ds_list.append(session_ds)

In [ ]:
output_ds = datasets.concatenate_datasets(session_ds_list)

In [ ]:
output_ds_path = os.path.join(output_ds_root, ds_key)
output_ds_path = output_ds_path.replace("ensemble", "ensemble_v2")
output_ds.save_to_disk(output_ds_path)

In [ ]:
session_ds_list[0][0]['img_raw']